In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from pathlib import Path
project_root = Path(os.path.dirname(os.getcwd()))
sys.path.append(str(project_root.joinpath('src')))

from utils.neuron_selectivity import corr_half
from utils.data_loader import sort_resp_by_train_test_trials, normalize_n_reorder_responses
from utils.neuron_selectivity import reliable_neurons
from utils.stimulus_loader import find_n_imgs
from utils.gain_model import fit_multiplicative_gain_svd, ev_additive_residuals_by_rank
from utils.decoder_functions import decode_all
from utils.equivariance_functions import example_equiv_projs, equivariance_summary_all
from utils.plotting_functions import plot_classification_errors, plot_orientation_errors, \
    plot_equivariance_variance, plot_equivariance_ccnorm, plot_equivariance_dim, \
    plot_equivariance_pcorr, plot_hist_lines, plot_points_w_errorbars


In [ ]:
model_df = pd.read_csv(project_root / 'params' / 'model_params.csv', header=0)
params_df = pd.read_csv(project_root / 'params' / 'fixed_params.csv', header=0, index_col=0)


### Estimate inv_snr from params/fixed_params.csv

In [ ]:
tx108_df = model_df[model_df['network'].eq('TX108_05_31_2023')]
model_name = tx108_df['network'].iloc[0]
layer_names = tx108_df['layer'].values
corr_thre = float(params_df.loc['corr_thre'].value)
cache_folder = project_root / f'cache' / model_name
print(f'Processing {model_name}: {list(layer_names)}')

In [ ]:

stim_name = params_df.loc['stimulus'].value
monitor = model_df.loc[model_df['network'].eq(model_name), 'monitor'].iloc[0]
data_root = project_root / 'data'
stim_root = data_root / 'stimuli'

# whole-vector split-half cc is cached so reruns do not reload the raw responses
cc_whole_f = cache_folder / f'{model_name}_cc_half_train_wholevec.csv'
if cc_whole_f.exists():
    cc_whole = np.loadtxt(cc_whole_f, delimiter=',')
else:
    date = model_name[-10:]
    db = np.load(data_root / date / f'spk_slants_{model_name}.npz', allow_pickle=True)
    resp_train_trials, _ = sort_resp_by_train_test_trials(
        db['spks'].T, db['istim'], stim_root, stim_name, monitor=monitor)
    resp_norm = normalize_n_reorder_responses(resp_train_trials, db['uspont'], db['mu'], db['sd'])
    n_rot, n_train, n_test = find_n_imgs(stim_name)
    resp_norm = resp_norm.reshape((resp_norm.shape[0], n_train, n_rot, resp_norm.shape[2]))
    idx_resp = reliable_neurons(resp_norm, cache_folder / f'{model_name}_cc_half_train_avg.csv', corr_thre)
    x = resp_norm[idx_resp]
    cc_whole = corr_half(x.reshape((x.shape[0], 1, -1, x.shape[-1]))).mean(-1).mean(-1)
    np.savetxt(cc_whole_f, cc_whole, delimiter=',')

# drop the few neurons whose whole-vector cc <= corr_thre (near-zero/negative cc would make the
# variance ratio (1-cc)/cc blow up or go negative)
cc_reliable = cc_whole[cc_whole > corr_thre]
inv_snr = ((1 - cc_reliable) / cc_reliable).mean()
print(f'TX108 reliable neurons: {cc_reliable.size} / {cc_whole.size}')
print('inv_snr = mean((1-cc)/cc) (train stimuli): %.2f' % inv_snr)
print('sqrt(inv_snr) (noise std factor): %.2f' % np.sqrt(inv_snr))

### Estimate noise correlations

In [ ]:
i0=6
stim_name = params_df.loc['stimulus'].value
corr_thre = float(params_df.loc['corr_thre'].value)

data_root = project_root / 'data'
stim_root = data_root / 'stimuli'
retinotopy_root = data_root / 'retinotopy'

model_name = model_df['network'].values[i0]
monitor = model_df['monitor'].values[i0]
print(f'Processing {model_name}')

cache_folder = project_root / f'cache_gain' / model_name
cache_folder.mkdir(parents=True, exist_ok=True)
results_folder_gain = project_root / 'results_extended_gain'
results_folder_gain.mkdir(parents=True, exist_ok=True)

In [ ]:
date = model_name[-10:]
db = np.load(data_root / date / f'spk_slants_{model_name}.npz', allow_pickle=True)

resp_train_trials, resp_test_trials = sort_resp_by_train_test_trials(db['spks'].T, db['istim'], \
                                                                    stim_root, stim_name, monitor=monitor)
resp_train_trials_norm = normalize_n_reorder_responses(resp_train_trials, db['uspont'], db['mu'], db['sd']) 
resp_test_trials_norm = normalize_n_reorder_responses(resp_test_trials, db['uspont'], db['mu'], db['sd']) 
xy_pos = np.array([db['xpos'],db['ypos']]).T

#reformat responses into (n_neurons, n_stim, n_rot, n_trials)
n_rot, n_train, n_test = find_n_imgs(stim_name)
resp_train_trials_norm = resp_train_trials_norm.reshape((resp_train_trials_norm.shape[0], \
                                                            n_train, n_rot, resp_train_trials_norm.shape[2]))
resp_test_trials_norm = resp_test_trials_norm.reshape((resp_test_trials_norm.shape[0], \
                                                            n_test, n_rot, resp_test_trials_norm.shape[2]))                                                           

#find reliably responsive neurons
cc_half_train_f = cache_folder / f'{model_name}_cc_half_train.csv'
idx_resp = reliable_neurons(resp_train_trials_norm, cc_half_train_f, corr_thre)

In [ ]:
# Spontaneous-projected but gain-preserved responses
resp_train_trials_spont = normalize_n_reorder_responses(resp_train_trials, db['uspont'], db['mu'], db['sd'], remove_gain=False)
resp_test_trials_spont = normalize_n_reorder_responses(resp_test_trials, db['uspont'], db['mu'], db['sd'], remove_gain=False)

resp_train_trials_spont = resp_train_trials_spont.reshape((resp_train_trials_spont.shape[0], \
                                                            n_train, n_rot, resp_train_trials_spont.shape[2]))
resp_test_trials_spont = resp_test_trials_spont.reshape((resp_test_trials_spont.shape[0], \
                                                            n_test, n_rot, resp_test_trials_spont.shape[2]))

# Raw responses (no normalization at all) for comparison plots
resp_train_trials2 = np.transpose(resp_train_trials,[2,1,0]).reshape((resp_train_trials_norm.shape[0], \
                                                                      n_train, n_rot, resp_train_trials_norm.shape[-1]))
resp_test_trials2 = np.transpose(resp_test_trials,[2,1,0]).reshape((resp_test_trials_norm.shape[0], \
                                                                      n_test, n_rot, resp_test_trials_norm.shape[-1]))

### Estimate global gain parameters

In [ ]:
x_train = resp_train_trials_spont[idx_resp].reshape((idx_resp.shape[0], -1, resp_train_trials_spont.shape[-1]))
a, g, f = fit_multiplicative_gain_svd(x_train)

In [ ]:
pd.DataFrame(a).to_csv(project_root / 'src' / 'utils' / 'gain_params' / 'a_train.csv', header=False, index=False)
pd.DataFrame(g).to_csv(project_root / 'src' / 'utils' / 'gain_params' / 'g_train.csv', header=False, index=False)

In [ ]:
x_test = resp_test_trials_spont[idx_resp].reshape((idx_resp.shape[0], -1, resp_test_trials_spont.shape[-1]))
a_test, g_test, f_test = fit_multiplicative_gain_svd(x_test)

In [ ]:
pd.DataFrame(a_test).to_csv(project_root / 'src' / 'utils' / 'gain_params' / 'a_test.csv', header=False, index=False)
pd.DataFrame(g_test).to_csv(project_root / 'src' / 'utils' / 'gain_params' / 'g_test.csv', header=False, index=False)

In [ ]:
hist_bins = np.linspace(-1, 1, 70)

# Build DataFrame of pairwise correlations
dfs = []
color_dict = {}
for label, x_train, color in [
    ('raw', resp_train_trials2[idx_resp], 'rgb(255,127,14)'),
    ('normalized', resp_train_trials_norm[idx_resp], 'rgb(31,119,180)'),
]:
    flat = x_train.reshape((x_train.shape[0], -1))
    corr_matrix = np.corrcoef(flat)
    upper = corr_matrix[np.triu_indices(flat.shape[0], k=1)]
    mean_corr = upper.mean()
    legend_label = f'{label} (mean={mean_corr:.3f})'
    # print(label, mean_corr)
    dfs.append(pd.DataFrame({'Pairwise correlation': upper, 'type': legend_label}))
    color_dict[legend_label] = color

df_corr = pd.concat(dfs, ignore_index=True)
color_df = pd.DataFrame({'color': color_dict})

fig = plot_hist_lines(df_corr, 'Pairwise correlation', 'type', hist_bins, color_df, height=300, width=400)
fig.update_layout(xaxis_range=[-0.2, 0.2], yaxis_title='Frac. neuron pairs')
fig.write_image(results_folder_gain / 'panel_a_noise_correlations.pdf', scale=1, engine='kaleido')
fig.show()

In [ ]:
for label, x_train, color, xlim, ylim, ticks in [
    ('raw', resp_train_trials2[idx_resp], 'rgba(255,127,14,0.1)', (0.001, 2.0), (0.001, 2.0),
     [0.001, 0.01, 0.1, 1]),
    ('normalized', resp_train_trials_norm[idx_resp], 'rgba(31,119,180,0.1)', (0.001, 30), (0.001, 30),
     [0.001, 0.01, 0.1, 1, 10]),
]:
    flat = x_train.reshape((x_train.shape[0], -1))
    means = flat.mean(axis=1)
    stds = flat.std(axis=1)

    df_plot = pd.DataFrame({
        'x': means, 'y': stds,
        'errx': 0.0, 'erry': 0.0,
        'color': color, 'errcolor': color,
    })

    tick_labels = [f'{v:g}' for v in ticks]
    fig = plot_points_w_errorbars(df_plot, sz=1, height=200, width=220, )
    fig.update_xaxes(title_text='Mean response', type='log',
                     range=[np.log10(xlim[0]), np.log10(xlim[1])],
                     title_standoff=0,
                     tickvals=ticks, ticktext=tick_labels)
    fig.update_yaxes(title_text='Std response', type='log',
                     range=[np.log10(ylim[0]), np.log10(ylim[1])],
                     title_standoff=0,
                     tickvals=ticks, ticktext=tick_labels)
    # Add unity line
    fig.add_shape(type='line', x0=xlim[0], y0=xlim[0], x1=xlim[1], y1=ylim[1],
                  line=dict(color='black', width=0.5, dash='dash'), opacity=0.3)
    # fig.update_layout(title=label)
    fig.write_image(results_folder_gain / f'panel_b_mean_vs_std_{label}.pdf')
    fig.show()

In [ ]:
# Cumulative EV of trial-to-trial noise by SVD rank (additive residuals)
x_train_3d = resp_train_trials_spont[idx_resp].reshape((idx_resp.shape[0], -1, resp_train_trials_spont.shape[-1]))
x_test_3d = resp_test_trials_spont[idx_resp].reshape((idx_resp.shape[0], -1, resp_test_trials_spont.shape[-1]))

max_rank = 20
ev_train = ev_additive_residuals_by_rank(x_train_3d, max_rank)
ev_test = ev_additive_residuals_by_rank(x_test_3d, max_rank)

print(f'Rank-1 EV of trial-to-trial noise — train: {ev_train[0]:.4f}, test: {ev_test[0]:.4f}')

In [ ]:
ranks = np.arange(1, max_rank + 1)
ranks_full = np.concatenate([[0], ranks])

color_train = 'rgba(0, 150, 136, 1.0)'   # teal
color_test = 'rgba(233, 30, 99, 1.0)'    # coral/pink

fig = go.Figure()
for ev, label, color, symb in [
    (ev_train, 'train', color_train, 'circle'),
    (ev_test, 'test', color_test, 'square'),
]:
    ev_full = np.concatenate([[0], ev])
    df_plot = pd.DataFrame({
        'x': ranks_full, 'y': ev_full,
        'errx': 0.0, 'erry': 0.0,
        'color': color, 'errcolor': color,
    })
    fig_i = plot_points_w_errorbars(df_plot, lines=True, sz=6, height=200, width=220)
    for trace in fig_i.data:
        trace.name = label
        trace.showlegend = True
        trace.marker.symbol = symb
        fig.add_trace(trace)

fig.update_layout(
    height=200, width=220,
    paper_bgcolor='rgba(255,255,255,255)',
    plot_bgcolor='rgba(255,255,255,0)',
    font={'size': 11, 'family': 'arial', 'color': 'black'},
    margin=dict(l=20, r=20, b=20, t=20, pad=0),
    legend=dict(x=0.45, y=0.15),
)
fig.update_xaxes(
    title_text='Number of components', showline=True, showgrid=False,
    linewidth=0.5, linecolor='black', mirror=True,
    title_standoff=0,
    tickfont={'size': 10, 'family': 'arial', 'color': 'black'},
)
fig.update_yaxes(
    title_text='Cumulative EV of<br>trial-to-trial noise',
    showline=True, showgrid=False,
    linewidth=0.5, linecolor='black', mirror=True,
    title_standoff=0,
    ticks='outside', ticklen=2, tickwidth=0.5,
    tickfont={'size': 10, 'family': 'arial', 'color': 'black'},
)
fig.write_image(results_folder_gain / 'panel_c_svd_explained_variance.pdf', scale=1, engine='kaleido')
fig.show()

### Apply global gain model to AlexNet

In [ ]:
results_folder_gain = project_root / 'results_extended_gain'
results_folder_gain.mkdir(parents=True, exist_ok=True)

#gain params: same pipeline as the main analysis but with the multiplicative-gain noise
#model and a reduced model list (pixels, AlexNet conv1-5, and mouse 1 as reference)
model_df_gain = pd.read_csv( project_root / 'params' / 'model_params_gain.csv', header=0)
params_df_gain = pd.read_csv( project_root / 'params' / 'fixed_params_gain.csv', header=0, index_col=0)

In [ ]:
### first run takes a long time
# re-runs AlexNet inference and the equivariance / decoding analyses under the
# multiplicative-gain noise model; every cache and figure gets a '_mult_gain' suffix
# (derived from noise_model in fixed_params_gain.csv) so the default gaussian results
# are not overwritten. Results are cached and reused on subsequent runs.
example_equiv_projs(model_df_gain, params_df_gain, project_root)
equivariance_summary_all(model_df_gain, params_df_gain, project_root)
decode_all(model_df_gain, params_df_gain, project_root, c_range=[0.001])

In [ ]:
#Fig 1d analogue: classification (texture) decoding error
# figure stored in project_root / 'results_extended_gain' / 'svm_0.001C_250avg_mult_gain.pdf'
plot_classification_errors(model_df_gain, params_df_gain, project_root, C0=0.001, symb='square', results_folder=results_folder_gain)

#Fig 1e analogue: absolute orientation estimation error
# figure stored in project_root / 'results_extended_gain' / 'ori_0ref_250avg_mult_gain.pdf'
plot_orientation_errors(model_df_gain, params_df_gain, project_root, symb='square', results_folder=results_folder_gain)

#Fig 3b bottom analogue: dimension of the invariant and tuned subspaces
# figure stored in project_root / 'results_extended_gain' / 'dim_250avg_20dim_mult_gain.pdf'
plot_equivariance_dim(model_df_gain, params_df_gain, project_root, results_folder=results_folder_gain)

#Fig 4b analogue: Pearson correlation of the invariant and tuned subspaces
# figure stored in project_root / 'results_extended_gain' / 'pcorr_250avg_20dim_mult_gain.pdf'
plot_equivariance_pcorr(model_df_gain, params_df_gain, project_root, results_folder=results_folder_gain)

#ED Fig 10b analogue: normalized correlation coefficient (CCnorm)
# figure stored in project_root / 'results_extended_gain' / 'cc_250avg_20dim_mult_gain.pdf'
plot_equivariance_ccnorm(model_df_gain, params_df_gain, project_root, show_mice=False, results_folder=results_folder_gain)

#ED Fig 10c analogue: variance explained by tuned and invariant dimensions
# figure stored in project_root / 'results_extended_gain' / 'var_250avg_20dim_mult_gain.pdf'
plot_equivariance_variance(model_df_gain, params_df_gain, project_root, show_mice=False, results_folder=results_folder_gain)